# Notebook 14 — Original FER baseline (N07 reuse)

Calculates one result for each of the 120 originals using frozen predictions already computed in N07. The analysis is **not yet executed**. Does not train a new classifier.

Out-of-set predictions such as `disgust` are kept as errors. Primary Macro-F1 averages 6 prespecified ground-truth classes; a 7-class auxiliary metric and confusion matrix are also produced.


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, json
from datetime import datetime, timezone
drive.mount('/content/drive')
RUNS=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=RUNS/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
assert N07.is_dir(), 'Frozen N07 run not found'
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
raw=pd.read_csv(N07/'raw_results.csv')
sources=pd.read_csv(N07/'selected_sources.csv')
assert len(raw)==600 and len(sources)==120 and sources.source_id.nunique()==120
assert (raw.emotion_status=='success').all()
out=RUNS/('nb14_original_fer_baseline_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
out.mkdir(parents=True,exist_ok=False)
print(out)


In [ ]:
norm=lambda s: s.astype(str).str.strip().str.lower().str.replace(' ','_',regex=False)
assert raw.groupby('source_id').original_predicted_class.nunique(dropna=False).eq(1).all(), 'Source predictions inconsistent'
assert raw.groupby('source_id').human_target_emotion.nunique(dropna=False).eq(1).all()
base=raw.groupby('source_id',as_index=False)[['original_predicted_class','human_target_emotion']].first()
joined=sources[['source_id','emotion']].merge(base,on='source_id',validate='one_to_one')
assert len(joined)==120
joined['ground_truth']=norm(joined.emotion)
joined['predicted']=norm(joined.original_predicted_class)
assert (joined.ground_truth==norm(joined.human_target_emotion)).all()
labels=sorted(joined.ground_truth.unique().tolist())
extra_predicted_labels = sorted(set(joined.predicted) - set(labels))
all_labels = sorted(set(labels) | set(joined.predicted))
print('Extra predicted labels (counted as errors):', extra_predicted_labels)
result={'status':'EXISTING_N07_CLASSIFIER_BASELINE_NOT_RETRAINED','n_sources':120,'accuracy':float(accuracy_score(joined.ground_truth,joined.predicted)),'macro_f1':float(f1_score(joined.ground_truth,joined.predicted,labels=labels,average='macro',zero_division=0)),'weighted_f1':float(f1_score(joined.ground_truth,joined.predicted,labels=labels,average='weighted',zero_division=0)),'evaluation_labels':labels,'all_observed_labels':all_labels,'out_of_set_predictions':joined.loc[~joined.predicted.isin(labels),'predicted'].value_counts().to_dict(),'macro_f1_all_observed_labels':float(f1_score(joined.ground_truth,joined.predicted,labels=all_labels,average='macro',zero_division=0))}
(out/'original_fer_baseline.json').write_text(json.dumps(result,indent=2))
pd.DataFrame(classification_report(joined.ground_truth,joined.predicted,labels=all_labels,output_dict=True,zero_division=0)).transpose().to_csv(out/'original_fer_per_class.csv')
pd.DataFrame(confusion_matrix(joined.ground_truth,joined.predicted,labels=all_labels),index=all_labels,columns=all_labels).to_csv(out/'original_fer_confusion.csv')
print(json.dumps(result,indent=2))
